In [ ]:
# Installation commands (run in Jupyter)
!pip install osmnx networkx folium ortools

# Import libraries
import osmnx as ox          # Download real-world street network data
import networkx as nx       # Handle the graph and compute shortest paths
import folium               # Create interactive maps
import numpy as np          # Handle numerical computations
from ortools.constraint_solver import routing_enums_pb2, pywrapcp  # OR-Tools core

print("All libraries imported successfully!")

All libraries imported successfully!


In [ ]:
# 1. Define the center point of Delhi (e.g., India Gate)
center_point = (28.6129, 77.2295)
radius = 2000  # 2 kilometers, in meters

# 2. Download the road network graph
print("Downloading the road network for a 2km radius in Delhi. Please wait...")
G = ox.graph_from_point(center_point, dist=radius, network_type='drive')
print(f"Road network downloaded! Nodes: {len(G.nodes)}, Edges: {len(G.edges)}")

# 3. (Optional) Visualize a quick overview of the network
# fig, ax = ox.plot_graph(G, figsize=(10, 10), node_color='red', node_size=10)

Road network downloaded! Nodes: 1381, Edges: 3014


In [ ]:
# 1. Get all nodes from the graph as candidates
all_nodes = list(G.nodes)
# Set a random seed for reproducible results
np.random.seed(42)

# 2. Randomly choose 1 depot and 25 delivery points (ensuring they are distinct)
depot_index = np.random.choice(all_nodes, 1)[0]
delivery_indices = np.random.choice([n for n in all_nodes if n != depot_index], 25, replace=False)

# 3. Put all locations in one list, with the depot first (index 0)
locations = [depot_index] + list(delivery_indices)
print(f"Selected 1 depot and {len(locations)-1} delivery points.")

# 4. Fetch the latitude/longitude coordinates for each selected node
node_coords = {node: (G.nodes[node]['y'], G.nodes[node]['x']) for node in locations}
print("Location coordinates are ready.")

Selected 1 depot and 25 delivery points.
Location coordinates are ready.


In [ ]:
import math

def compute_distance_matrix(G, locations):
    """Calculate distance matrix with fallback to Euclidean distance."""
    n = len(locations)
    dist_matrix = np.zeros((n, n), dtype=np.int64)

    # Pre-compute coordinates for all locations
    coords = []
    for node in locations:
        coords.append((G.nodes[node]['y'], G.nodes[node]['x']))

    def euclidean_distance(i, j):
        """Straight-line distance in meters (approximate)"""
        lat1, lon1 = coords[i]
        lat2, lon2 = coords[j]
        # Rough conversion: 1 degree ≈ 111,000 meters
        return int(((lat1 - lat2)**2 + (lon1 - lon2)**2)**0.5 * 111000)

    for i in range(n):
        for j in range(n):
            if i == j:
                dist_matrix[i][j] = 0
            else:
                try:
                    path_length = nx.shortest_path_length(G, locations[i], locations[j], weight='length')
                    dist_matrix[i][j] = int(path_length)
                except nx.NetworkXNoPath:
                    # Fallback to Euclidean distance
                    dist_matrix[i][j] = euclidean_distance(i, j)
                    print(f"⚠️ No road path found between {i} and {j}, using Euclidean: {dist_matrix[i][j]}m")

    return dist_matrix

print("Computing the distance matrix...")
distance_matrix = compute_distance_matrix(G, locations)
print("Distance matrix computation complete!")
print(f"Distance matrix shape: {distance_matrix.shape}")
# Display the first 5x5 entries as a preview
print(distance_matrix[:5, :5])

Computing the distance matrix...
⚠️ No road path found between 0 and 9, using Euclidean: 2473m
⚠️ No road path found between 1 and 9, using Euclidean: 2319m
⚠️ No road path found between 2 and 9, using Euclidean: 1593m
⚠️ No road path found between 3 and 9, using Euclidean: 3117m
⚠️ No road path found between 4 and 9, using Euclidean: 3045m
⚠️ No road path found between 5 and 9, using Euclidean: 3539m
⚠️ No road path found between 6 and 9, using Euclidean: 734m
⚠️ No road path found between 7 and 9, using Euclidean: 2893m
⚠️ No road path found between 8 and 9, using Euclidean: 2261m
⚠️ No road path found between 10 and 9, using Euclidean: 355m
⚠️ No road path found between 11 and 9, using Euclidean: 276m
⚠️ No road path found between 12 and 9, using Euclidean: 2863m
⚠️ No road path found between 13 and 9, using Euclidean: 2144m
⚠️ No road path found between 14 and 9, using Euclidean: 2740m
⚠️ No road path found between 15 and 9, using Euclidean: 1643m
⚠️ No road path found between 16 a

In [ ]:
# Generate random demands for each location (0 for the depot)
np.random.seed(42)  # Keep it reproducible
demands = [0]  # Depot has no demand
for i in range(1, len(locations)):
    demands.append(np.random.randint(1, 10))  # Random demand between 1 and 9

print(f"Demands per location: {demands}")
print(f"Total demand: {sum(demands)}")

# Define capacity for each of the 3 trucks
# Set to 25 so that 1 truck cannot carry all (total ~55), forcing the use of at least 3 trucks.
vehicle_capacities = [25, 25, 25]
print(f"Truck capacities: {vehicle_capacities}")

Demands per location: [0, 7, 4, 8, 5, 7, 3, 7, 8, 5, 4, 8, 8, 3, 6, 5, 2, 8, 6, 2, 5, 1, 6, 9, 1, 3]
Total demand: 131
Truck capacities: [25, 25, 25]


In [ ]:
def solve_vrp_with_capacity(distance_matrix, demands, vehicle_capacities, num_vehicles=3, depot=0,time_limit_seconds=5):
    """
    Solve the VRP with capacity constraints.

    Args:
        distance_matrix: 2D list of ints (n x n)
        demands: list of ints, length = n (0 for depot)
        vehicle_capacities: list of ints, length = num_vehicles
        num_vehicles: int
        depot: int (index of depot)

    Returns:
        dict: vehicle_id -> list of node indices (includes depot at start and end)
    """
    # 1. Build data dictionary
    data = {}
    data['distance_matrix'] = distance_matrix
    data['demands'] = demands
    data['vehicle_capacities'] = vehicle_capacities
    data['num_vehicles'] = num_vehicles
    data['depot'] = depot

    # 2. Index Manager (only three arguments!)
    manager = pywrapcp.RoutingIndexManager(
        len(data['distance_matrix']),
        data['num_vehicles'],
        data['depot']
    )

    # 3. Routing Model
    routing = pywrapcp.RoutingModel(manager)

    # 4. Distance callback
    def distance_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        return data['distance_matrix'][from_node][to_node]

    transit_callback_index = routing.RegisterTransitCallback(distance_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    # 5. Demand callback
    def demand_callback(from_index):
        from_node = manager.IndexToNode(from_index)
        return data['demands'][from_node]

    demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)

    # 6. Add Capacity dimension
    routing.AddDimensionWithVehicleCapacity(
        demand_callback_index,
        0,  # no slack
        data['vehicle_capacities'],  # max load per vehicle
        True,  # start cumul to zero
        'Capacity'
    )

    # 7. Search parameters
    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    )

    # 🔥 NEW: Enable local search with time limit
    search_parameters.local_search_metaheuristic = (
        routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    )
    search_parameters.time_limit.seconds = time_limit_seconds

    # 8. Solve
    solution = routing.SolveWithParameters(search_parameters)

    if not solution:
        print("No solution found!")
        return None

    # 9. Extract routes
    routes = {}
    for vehicle_id in range(data['num_vehicles']):
        index = routing.Start(vehicle_id)
        route_nodes = []
        while not routing.IsEnd(index):
            node_index = manager.IndexToNode(index)
            route_nodes.append(node_index)
            index = solution.Value(routing.NextVar(index))
        route_nodes.append(manager.IndexToNode(index))  # back to depot
        routes[vehicle_id] = route_nodes

    return routes

# Generate demands (0 for depot)
np.random.seed(42)
demands = [0] + [np.random.randint(1, 10) for _ in range(len(locations)-1)]   # 20 deliveries

# Capacities for 3 trucks
vehicle_capacities = [50, 60, 65]   # adjust as needed

# Now call the solver
solution_routes = solve_vrp_with_capacity(
    distance_matrix=distance_matrix,
    demands=demands,
    vehicle_capacities=vehicle_capacities,
    num_vehicles=3,
    depot=0,
    time_limit_seconds=10
)

if solution_routes:
    total_distance = 0
    for v_id, route in solution_routes.items():
        load = sum(demands[node] for node in route)
        # Calculate distance for this truck
        truck_dist = sum(distance_matrix[route[i]][route[i+1]] for i in range(len(route)-1))
        total_distance += truck_dist
        print(f"Truck {v_id+1}: {route}  (load: {load}) - Distance: {truck_dist:,}m")

    print("\n" + "="*40)
    print(f"📏 TOTAL COMBINED DISTANCE: {total_distance:,} meters")
    print("="*40)

Truck 1: [0, 25, 24, 17, 0]  (load: 12) - Distance: 5,544m
Truck 2: [0, 13, 14, 23, 4, 3, 21, 5, 12, 7, 0]  (load: 54) - Distance: 10,897m
Truck 3: [0, 8, 22, 16, 19, 20, 11, 9, 10, 6, 18, 2, 15, 1, 0]  (load: 65) - Distance: 13,232m

📏 TOTAL COMBINED DISTANCE: 29,673 meters


In [ ]:
def visualize_routes_on_map(G, routes, locations, node_coords):
    """Plot the routes on an interactive Folium map using CartoDB tiles (avoid OSM blocking)."""
    # 1. Create a base map centered on the depot
    depot_osm_id = locations[0]
    depot_lat, depot_lon = node_coords[depot_osm_id]

    # 🔥 FIX: Use 'CartoDB positron' instead of 'OpenStreetMap'
    m = folium.Map(location=[depot_lat, depot_lon], zoom_start=14, tiles='CartoDB positron')

    # 2. Color palette for different trucks
    colors = ['red', 'blue', 'green', 'purple', 'orange', 'darkred',
              'lightred', 'beige', 'darkblue', 'darkgreen', 'cadetblue']

    # 3. Mark all locations
    for idx, osm_id in enumerate(locations):
        lat, lon = node_coords[osm_id]
        if idx == 0:  # Depot
            folium.Marker([lat, lon], popup="🏭 Depot",
                          icon=folium.Icon(color='black', icon='home')).add_to(m)
        else:         # Delivery points
            folium.Marker([lat, lon], popup=f"📍 Location {idx}",
                          icon=folium.Icon(color='gray', icon='info-sign')).add_to(m)

    # 4. Draw each truck's route as a polyline
    for vehicle_id, route_indices in routes.items():
        color = colors[vehicle_id % len(colors)]
        route_coords = []
        for idx in route_indices:
            osm_id = locations[idx]   # map index -> OSM node ID
            lat, lon = node_coords[osm_id]
            route_coords.append((lat, lon))

        folium.PolyLine(route_coords, color=color, weight=4, opacity=0.7,
                        popup=f"🚚 Truck {vehicle_id+1}").add_to(m)

    return m

# Generate the map
route_map = visualize_routes_on_map(G, solution_routes, locations, node_coords)

# Display the map in the notebook
route_map